# ?? ?? ???

- ??: ??? ?? ?? ?? ??? ?????.
- ??: ?? ???? ?? ??
- ??: ??? ?? ??
- ?? ??: ?? ML? ???? ?? ?? ?????.


In [ ]:
import pandas as pd

# 데이터 불러오기

In [ ]:
df = pd.read_excel('./data/final_all_data.xlsx')
df

In [ ]:
df.columns

In [ ]:
dates = ['22_08', '22_09', '22_10', '22_11', '22_12', '23_01', '23_02', '23_03', '23_04', '23_05', '23_06', '23_07', '23_08']

In [ ]:
vars = ['구독자점수', '조회수점수', '충성도점수', '구독자 성장율점수', '감성점수점수', '평균_영상간격점수',
       '평균 영상 좋아요 수 점수', '영상개수 점수', '등급' ]

In [ ]:
cols = []
for date in dates:
    for var in vars:
        cols.append(date + '_' + var)
cols

# 데이터프레임 형태 변환

In [ ]:
df.columns

In [ ]:
list(df.columns).index('구독자점수')

In [ ]:
df_new = pd.DataFrame(columns=['유투버'] + cols)
for name_idx, name in enumerate(df['유투버'].unique()):
    df_new.loc[name_idx, '유투버'] = name
    for i, date in enumerate(dates):
        start_name = df_new.columns[i*9+1]
        end_name = df_new.columns[(i+1)*9]
        df_new.loc[name_idx, start_name:end_name] = df[df['유투버'] == name].iloc[i, [15, 16, 17, 18, 19, 20, 21, 22, -1]].values

In [ ]:
df_new

In [ ]:
df_new.columns

In [ ]:
df_new.isnull().sum().sum()

# 3개월 데이터만 사용

## 등급 자체를 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat(( df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_train = pd.concat(( df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)


In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_train)
scaled_features = pd.DataFrame(scaled_features, columns = X_train.columns)
# 데이터 변환

X_tr= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_train.columns[:8].str.replace('15년_', ''))
    for j in range(3):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_tr.append(temp_df.values)

X_tr= np.array(X_tr)
y_tr = np.array(y_train.values)
X_tr.shape, y_tr.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_test)
scaled_features = pd.DataFrame(scaled_features, columns = X_test.columns)
# 데이터 변환

X_te= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_test.columns[:8].str.replace('15년_', ''))
    for j in range(3):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_te.append(temp_df.values)

X_te= np.array(X_te)
y_te = np.array(y_test.values)
X_te.shape, y_te.shape

In [ ]:
def label_encode_dataframe(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 1, 'B+': 2, 'B0': 3,
        'C+': 4, 'C0': 5, 'D+': 6, 'D0': 7,
        'E': 8, 'F': 9
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe(y_train, y_train.columns[0])
label_encode_dataframe(y_test, y_test.columns[0])

In [ ]:
from keras.utils import np_utils
y_train = np_utils.to_categorical(y_train)
y_test = np_utils.to_categorical(y_test)

In [ ]:
X_tr.shape

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# 시계열 데이터 생성 (예제 데이터)
# X: 시계열 특성 데이터, y: 다중 클래스 레이블
# 이 예제에서는 임의의 데이터를 생성합니다. 실제 데이터에 대한 적용을 위해 데이터를 준비해야 합니다.

X_tr = tf.convert_to_tensor(X_tr, dtype=tf.float32)
X_te = tf.convert_to_tensor(X_te, dtype=tf.float32)

# LSTM 모델 생성
model = Sequential()
# model.add(LSTM(units=128, input_shape=(3, 8), return_sequences=True))
model.add(LSTM(units=64,  input_shape=(3, 8)))
model.add(Dense(units=10, activation='softmax'))

# 모델 컴파일
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 조기 종료 콜백 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 저장 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, save_weights_only=False, monitor='val_loss', mode='min', verbose=1)

# 모델 학습
history = model.fit(X_tr, y_train, epochs=100, batch_size=16, validation_data=(X_te, y_test), callbacks=[early_stopping, model_checkpoint])

# 모델 평가
loss, accuracy = model.evaluate(X_te, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = model.predict(X_te)


y_pred_classes = np.argmax(predictions, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# 정확도 계산
accuracy = accuracy_score(y_true_classes, y_pred_classes)

# 정밀도 계산
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')

# 재현율 계산
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')

# F1 스코어 계산
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)


## 간소화된 등급 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat(( df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_train = pd.concat(( df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)


In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_train)
scaled_features = pd.DataFrame(scaled_features, columns = X_train.columns)
# 데이터 변환

X_tr= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_train.columns[:8].str.replace('15년_', ''))
    for j in range(3):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_tr.append(temp_df.values)

X_tr= np.array(X_tr)
y_tr = np.array(y_train.values)
X_tr.shape, y_tr.shape

In [ ]:
y_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_test)
scaled_features = pd.DataFrame(scaled_features, columns = X_test.columns)
# 데이터 변환

X_te= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_test.columns[:8].str.replace('15년_', ''))
    for j in range(3):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_te.append(temp_df.values)

X_te= np.array(X_te)
y_te = np.array(y_test.values)
X_te.shape, y_te.shape

In [ ]:
def label_encode_dataframe_2(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 1, 'B0': 1,
        'C+': 2, 'C0': 2, 'D+': 3, 'D0': 3,
        'E': 4, 'F': 4
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_2(y_train, y_train.columns[0])
label_encode_dataframe_2(y_test, y_test.columns[0])

In [ ]:
y_train = np_utils.to_categorical(y_train)
y_test = np_utils.to_categorical(y_test)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# 시계열 데이터 생성 (예제 데이터)
# X: 시계열 특성 데이터, y: 다중 클래스 레이블
# 이 예제에서는 임의의 데이터를 생성합니다. 실제 데이터에 대한 적용을 위해 데이터를 준비해야 합니다.

X_tr = tf.convert_to_tensor(X_tr, dtype=tf.float32)
X_te = tf.convert_to_tensor(X_te, dtype=tf.float32)

# LSTM 모델 생성
model = Sequential()
# model.add(LSTM(units=128, input_shape=(3, 8), return_sequences=True))
model.add(LSTM(units=64,  input_shape=(3, 8)))
model.add(Dense(units=5, activation='softmax'))

# 모델 컴파일
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 조기 종료 콜백 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 저장 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, save_weights_only=False, monitor='val_loss', mode='min', verbose=1)

# 모델 학습
history = model.fit(X_tr, y_train, epochs=100, batch_size=16, validation_data=(X_te, y_test), callbacks=[early_stopping, model_checkpoint])

# 모델 평가
loss, accuracy = model.evaluate(X_te, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = model.predict(X_te)


y_pred_classes = np.argmax(predictions, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# 정확도 계산
accuracy = accuracy_score(y_true_classes, y_pred_classes)

# 정밀도 계산
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')

# 재현율 계산
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')

# F1 스코어 계산
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)


## 인기, 비인기 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat(( df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_train = pd.concat(( df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_train)
scaled_features = pd.DataFrame(scaled_features, columns = X_train.columns)
# 데이터 변환

X_tr= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_train.columns[:8].str.replace('15년_', ''))
    for j in range(3):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_tr.append(temp_df.values)

X_tr= np.array(X_tr)
y_tr = np.array(y_train.values)
X_tr.shape, y_tr.shape

In [ ]:
y_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_test)
scaled_features = pd.DataFrame(scaled_features, columns = X_test.columns)
# 데이터 변환

X_te= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_test.columns[:8].str.replace('15년_', ''))
    for j in range(3):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_te.append(temp_df.values)

X_te= np.array(X_te)
y_te = np.array(y_test.values)
X_te.shape, y_te.shape

In [ ]:
def label_encode_dataframe_3(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 0, 'B0': 0,
        'C+': 1, 'C0': 1, 'D+': 1, 'D0': 1,
        'E': 1, 'F': 1
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_3(y_train, y_train.columns[0])
label_encode_dataframe_3(y_test, y_test.columns[0])

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# 시계열 데이터 생성 (예제 데이터)
# X: 시계열 특성 데이터, y: 다중 클래스 레이블
# 이 예제에서는 임의의 데이터를 생성합니다. 실제 데이터에 대한 적용을 위해 데이터를 준비해야 합니다.

X_tr = tf.convert_to_tensor(X_tr, dtype=tf.float32)
X_te = tf.convert_to_tensor(X_te, dtype=tf.float32)

# LSTM 모델 생성
model = Sequential()
# model.add(LSTM(units=128, input_shape=(3, 8), return_sequences=True))
model.add(LSTM(units=64,  input_shape=(3, 8)))
model.add(Dense(units=1, activation='sigmoid'))

# 모델 컴파일
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 조기 종료 콜백 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 저장 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, save_weights_only=False, monitor='val_loss', mode='min', verbose=1)

# 모델 학습
history = model.fit(X_tr, y_train, epochs=100, batch_size=16, validation_data=(X_te, y_test), callbacks=[early_stopping, model_checkpoint])

# 모델 평가
loss, accuracy = model.evaluate(X_te, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = model.predict(X_te)


y_pred = np.round(predictions)

# 정확도 계산
accuracy = accuracy_score(y_test, y_pred)

# 정밀도 계산
precision = precision_score(y_test, y_pred)

# 재현율 계산
recall = recall_score(y_test, y_pred)

# F1 스코어 계산
f1 = f1_score(y_test, y_pred)

print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)


# 데이터 전부 사용

## 등급 자체를 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -108:-100], df_new.iloc[:, -99:-91], df_new.iloc[:, -90:-82], df_new.iloc[:, -81:-73], df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = pd.concat((df_new.iloc[:, -117:-109], df_new.iloc[:, -108:-100], df_new.iloc[:, -99:-91], df_new.iloc[:, -90:-82], df_new.iloc[:, -81:-73], df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_train)
scaled_features = pd.DataFrame(scaled_features, columns = X_train.columns)
# 데이터 변환

X_tr= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_train.columns[:8].str.replace('15년_', ''))
    for j in range(11):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_tr.append(temp_df.values)

X_tr= np.array(X_tr)
y_tr = np.array(y_train.values)
X_tr.shape, y_tr.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_test)
scaled_features = pd.DataFrame(scaled_features, columns = X_test.columns)
# 데이터 변환

X_te= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_test.columns[:8].str.replace('15년_', ''))
    for j in range(11):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_te.append(temp_df.values)

X_te= np.array(X_te)
y_te = np.array(y_test.values)
X_te.shape, y_te.shape

In [ ]:
def label_encode_dataframe(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 1, 'B+': 2, 'B0': 3,
        'C+': 4, 'C0': 5, 'D+': 6, 'D0': 7,
        'E': 8, 'F': 9
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe(y_train, y_train.columns[0])
label_encode_dataframe(y_test, y_test.columns[0])

In [ ]:
y_train = np_utils.to_categorical(y_train)
y_test = np_utils.to_categorical(y_test)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# 시계열 데이터 생성 (예제 데이터)
# X: 시계열 특성 데이터, y: 다중 클래스 레이블
# 이 예제에서는 임의의 데이터를 생성합니다. 실제 데이터에 대한 적용을 위해 데이터를 준비해야 합니다.

X_tr = tf.convert_to_tensor(X_tr, dtype=tf.float32)
X_te = tf.convert_to_tensor(X_te, dtype=tf.float32)

# LSTM 모델 생성
model = Sequential()
# model.add(LSTM(units=128, input_shape=(11, 8), return_sequences=True))
model.add(LSTM(units=64,  input_shape=(11, 8)))
model.add(Dense(units=10, activation='softmax'))

# 모델 컴파일
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 조기 종료 콜백 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 저장 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, save_weights_only=False, monitor='val_loss', mode='min', verbose=1)

# 모델 학습
history = model.fit(X_tr, y_train, epochs=100, batch_size=16, validation_data=(X_te, y_test), callbacks=[early_stopping, model_checkpoint])

# 모델 평가
loss, accuracy = model.evaluate(X_te, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = model.predict(X_te)


y_pred_classes = np.argmax(predictions, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# 정확도 계산
accuracy = accuracy_score(y_true_classes, y_pred_classes)

# 정밀도 계산
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')

# 재현율 계산
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')

# F1 스코어 계산
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)


# 간소화된 등급 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -108:-100], df_new.iloc[:, -99:-91], df_new.iloc[:, -90:-82], df_new.iloc[:, -81:-73], df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = pd.concat((df_new.iloc[:, -117:-109], df_new.iloc[:, -108:-100], df_new.iloc[:, -99:-91], df_new.iloc[:, -90:-82], df_new.iloc[:, -81:-73], df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_train)
scaled_features = pd.DataFrame(scaled_features, columns = X_train.columns)
# 데이터 변환

X_tr= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_train.columns[:8].str.replace('15년_', ''))
    for j in range(11):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_tr.append(temp_df.values)

X_tr= np.array(X_tr)
y_tr = np.array(y_train.values)
X_tr.shape, y_tr.shape

In [ ]:
y_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_test)
scaled_features = pd.DataFrame(scaled_features, columns = X_test.columns)
# 데이터 변환

X_te= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_test.columns[:8].str.replace('15년_', ''))
    for j in range(11):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_te.append(temp_df.values)

X_te= np.array(X_te)
y_te = np.array(y_test.values)
X_te.shape, y_te.shape

In [ ]:
def label_encode_dataframe_2(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 1, 'B0': 1,
        'C+': 2, 'C0': 2, 'D+': 3, 'D0': 3,
        'E': 4, 'F': 4
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_2(y_train, y_train.columns[0])
label_encode_dataframe_2(y_test, y_test.columns[0])

In [ ]:
y_train = np_utils.to_categorical(y_train)
y_test = np_utils.to_categorical(y_test)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# 시계열 데이터 생성 (예제 데이터)
# X: 시계열 특성 데이터, y: 다중 클래스 레이블
# 이 예제에서는 임의의 데이터를 생성합니다. 실제 데이터에 대한 적용을 위해 데이터를 준비해야 합니다.

X_tr = tf.convert_to_tensor(X_tr, dtype=tf.float32)
X_te = tf.convert_to_tensor(X_te, dtype=tf.float32)

# LSTM 모델 생성
model = Sequential()
# model.add(LSTM(units=128, input_shape=(11, 8), return_sequences=True))
model.add(LSTM(units=64,  input_shape=(11, 8)))
model.add(Dense(units=5, activation='softmax'))

# 모델 컴파일
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 조기 종료 콜백 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 저장 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, save_weights_only=False, monitor='val_loss', mode='min', verbose=1)

# 모델 학습
history = model.fit(X_tr, y_train, epochs=100, batch_size=16, validation_data=(X_te, y_test), callbacks=[early_stopping, model_checkpoint])

# 모델 평가
loss, accuracy = model.evaluate(X_te, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = model.predict(X_te)


y_pred_classes = np.argmax(predictions, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# 정확도 계산
accuracy = accuracy_score(y_true_classes, y_pred_classes)

# 정밀도 계산
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')

# 재현율 계산
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')

# F1 스코어 계산
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)


# 인기, 비인기 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -108:-100], df_new.iloc[:, -99:-91], df_new.iloc[:, -90:-82], df_new.iloc[:, -81:-73], df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = pd.concat((df_new.iloc[:, -117:-109], df_new.iloc[:, -108:-100], df_new.iloc[:, -99:-91], df_new.iloc[:, -90:-82], df_new.iloc[:, -81:-73], df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_train)
scaled_features = pd.DataFrame(scaled_features, columns = X_train.columns)
# 데이터 변환

X_tr= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_train.columns[:8].str.replace('15년_', ''))
    for j in range(11):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_tr.append(temp_df.values)

X_tr= np.array(X_tr)
y_tr = np.array(y_train.values)
X_tr.shape, y_tr.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(X_test)
scaled_features = pd.DataFrame(scaled_features, columns = X_test.columns)
# 데이터 변환

X_te= []
for i in range(len(scaled_features)):
    temp_df = pd.DataFrame(columns = X_test.columns[:8].str.replace('15년_', ''))
    for j in range(11):
        temp_df.loc[j, :] = scaled_features.iloc[i, j*8:(j+1)*8].values

    X_te.append(temp_df.values)

X_te= np.array(X_te)
y_te = np.array(y_test.values)
X_te.shape, y_te.shape

In [ ]:
def label_encode_dataframe_3(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 0, 'B0': 0,
        'C+': 1, 'C0': 1, 'D+': 1, 'D0': 1,
        'E': 1, 'F': 1
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_3(y_train, y_train.columns[0])
label_encode_dataframe_3(y_test, y_test.columns[0])

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# 시계열 데이터 생성 (예제 데이터)
# X: 시계열 특성 데이터, y: 다중 클래스 레이블
# 이 예제에서는 임의의 데이터를 생성합니다. 실제 데이터에 대한 적용을 위해 데이터를 준비해야 합니다.

X_tr = tf.convert_to_tensor(X_tr, dtype=tf.float32)
X_te = tf.convert_to_tensor(X_te, dtype=tf.float32)

# LSTM 모델 생성
model = Sequential()
#model.add(LSTM(units=128, input_shape=(11, 8), return_sequences=True))
model.add(LSTM(units=64,  input_shape=(11, 8)))
model.add(Dense(units=1, activation='sigmoid'))

# 모델 컴파일
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 조기 종료 콜백 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 저장 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, save_weights_only=False, monitor='val_loss', mode='min', verbose=1)

# 모델 학습
history = model.fit(X_tr, y_train, epochs=100, batch_size=16, validation_data=(X_te, y_test), callbacks=[early_stopping, model_checkpoint])

# 모델 평가
loss, accuracy = model.evaluate(X_te, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = model.predict(X_te)


y_pred = np.round(predictions)

# 정확도 계산
accuracy = accuracy_score(y_test, y_pred)

# 정밀도 계산
precision = precision_score(y_test, y_pred)

# 재현율 계산
recall = recall_score(y_test, y_pred)

# F1 스코어 계산
f1 = f1_score(y_test, y_pred)

print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
